# Analysis

In [1]:
# Cell 0: Setup
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

if Path('/content/data').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import importlib
import src.gap_analysis
importlib.reload(src.gap_analysis)

<module 'src.gap_analysis' from '/Users/trishasalas/Repos/Research/tmlr/src/gap_analysis.py'>

In [3]:
# In analysis.ipynb:
from src.gap_analysis import run_gap_analysis, save_tables, print_summary

tables = run_gap_analysis(PROJECT_ROOT)
save_tables(tables, PROJECT_ROOT / 'results' / 'analysis')
print_summary(tables)

Loaded:
  Elicitation: 3458 rows across 13 models (['gpt2', 'olmo', 'pythia'])  source={'original': 2925, 'expansion': 533}
  Entropy: 3284 rows across 13 models (['gpt2', 'olmo', 'pythia'])  source={'original': 2751, 'expansion': 533}
  Binding: 2150144 rows across 13 models (['gpt2', 'olmo', 'pythia'])  source={'original': 1723904, 'expansion': 426240}

Skipped 13 CSV(s) — stem did not resolve to a KNOWN_DOMAIN:
  results/entropy/gpt2/gpt2-large/gpt2-large-entropy.csv  ->  domain='entropy'
  results/entropy/gpt2/gpt2-medium/gpt2-medium-entropy.csv  ->  domain='entropy'
  results/entropy/gpt2/gpt2-small/gpt2-entropy.csv  ->  domain='gpt2-entropy'
  results/entropy/gpt2/gpt2-xl/gpt2-xl-entropy.csv  ->  domain='entropy'
  results/entropy/olmo/OLMo-2-0425-1B/OLMo-2-0425-1B-entropy.csv  ->  domain='entropy'
  results/entropy/olmo/OLMo-2-1124-13B/OLMo-2-1124-13B-entropy.csv  ->  domain='entropy'
  results/entropy/olmo/OLMo-2-1124-7B/OLMo-2-1124-7B-entropy.csv  ->  domain='entropy'
  result

## Extended analyses

Six analyses built on the existing entropy/binding/results CSVs (no new experiments).
All tables are produced by `run_gap_analysis` and saved to `results/analysis/`.

1. **Fluent wrongness** — confidence (last-token entropy) when wrong on accessibility vs right on the bicycle control
2. **Binding depth vs accuracy** — max binding score paired with declarative accuracy, with correlation
3. **Per-concept scaling curves** — accuracy trajectory per concept, classified (climb / peak-regress / never-emerges)
4. **Declarative vs evaluative entropy divergence** — internal uncertainty on the task the model behaviorally fails
5. **Degenerate output detection** — rate of repetitive collapse by scale / concept / prompt type
6. **Completion paradox** — few-shot syntactic completion accuracy vs declarative accuracy

In [4]:
# Display the extended-analysis tables (produced above in `tables`)
from IPython.display import display

def show(name, caption):
    print(f"\n{'='*70}\n{caption}\n{'='*70}")
    display(tables[name])

# Ext 1 — fluent wrongness (negative confidence_gap = more confident when wrong)
show('fluent_wrongness', 'Ext 1 — Fluent wrongness (a11y-incorrect vs control-correct entropy)')

# Ext 2 — binding depth vs accuracy + correlation
show('binding_accuracy_corr', 'Ext 2 — Binding depth vs accuracy: correlation per suite')
show('binding_vs_accuracy', 'Ext 2 — Binding depth vs accuracy: per compound x scale')

# Ext 3 — per-concept trajectories
show('per_concept_trajectories', 'Ext 3 — Per-concept scaling trajectories')

# Ext 4 — declarative vs evaluative entropy divergence
show('entropy_divergence', 'Ext 4 — Declarative vs evaluative entropy divergence')

# Ext 5 — degenerate output rates
show('degenerate_by_scale', 'Ext 5 — Degenerate output rate by scale')
show('degenerate_by_prompt_type', 'Ext 5 — Degenerate output rate by prompt type')

# Ext 6 — completion paradox
show('completion_paradox', 'Ext 6 — Completion paradox (completion% vs declarative%)')


Ext 1 — Fluent wrongness (a11y-incorrect vs control-correct entropy)


,suite,scale,n_access_incorrect,access_incorrect_entropy,n_control_correct,control_correct_entropy,confidence_gap
0,pythia,160M,38,4.3233,4,3.1541,1.1691
1,pythia,410M,36,3.9213,4,3.2012,0.7200
2,pythia,1B,40,3.7176,3,3.0190,0.6986
3,pythia,2.8B,32,3.6732,5,3.2593,0.4139
4,pythia,6.9B,30,3.6182,5,3.2919,0.3263
5,pythia,12B,23,3.5972,5,3.0655,0.5316
6,gpt2,124M,36,4.7010,2,3.9656,0.7354
7,gpt2,355M,31,4.5171,4,3.9990,0.5180
8,gpt2,774M,38,3.9531,4,3.6654,0.2877
9,gpt2,1.5B,30,3.5251,5,3.4310,0.0941



Ext 2 — Binding depth vs accuracy: correlation per suite


,suite,n_pairs,pearson_r,spearman_r
0,gpt2,196,0.087,-0.059
1,olmo,147,0.115,0.108
2,pythia,294,-0.003,-0.126



Ext 2 — Binding depth vs accuracy: per compound x scale


...

,suite,scale_label,n,n_degenerate,pct_degenerate
0,gpt2,1.5B,266,4,1.5
1,gpt2,124M,266,16,6.0
2,gpt2,355M,266,10,3.8
3,gpt2,774M,266,5,1.9
4,olmo,13B,266,5,1.9
5,olmo,1B,266,6,2.3
6,olmo,7B,266,7,2.6
7,pythia,12B,266,6,2.3
8,pythia,160M,266,16,6.0
9,pythia,1B,266,13,4.9



Ext 5 — Degenerate output rate by prompt type


,prompt_type,n,n_degenerate,pct_degenerate
0,completion,104,33,31.7
1,control,260,4,1.5
2,declarative,2925,51,1.7
3,evaluative,65,7,10.8
4,hypothesis,39,10,25.6
5,validation,65,5,7.7



Ext 6 — Completion paradox (completion% vs declarative%)


,suite,concept,scale,completion_pct,declarative_pct,paradox_gap_pct_pts
0,pythia,accessibility_tree,160M,NaN,50.0,NaN
1,pythia,accessibility_tree,410M,NaN,50.0,NaN
2,pythia,accessibility_tree,1B,NaN,50.0,NaN
3,pythia,accessibility_tree,2.8B,NaN,50.0,NaN
4,pythia,accessibility_tree,6.9B,NaN,50.0,NaN
...,...,...,...,...,...,...
684,olmo,universal_design,7B,NaN,0.0,NaN
685,olmo,universal_design,13B,NaN,0.0,NaN
686,olmo,wcag,1B,NaN,100.0,NaN
687,olmo,wcag,7B,NaN,100.0,NaN
